In [6]:
import yfinance as yf
import pandas as pd
from newsapi import NewsApiClient
from datetime import datetime, timedelta

end_date = datetime.today()
start_date = end_date - timedelta(days=30)

ticker = yf.Ticker("GOOG")

df_prices = ticker.history(
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    auto_adjust=True
)

df_prices = df_prices[["Close", "Volume"]]
df_prices.index = pd.to_datetime(df_prices.index).tz_localize(None).normalize()
df_prices.index = df_prices.index.date

print(f"Prices obtained: {len(df_prices)} days")
print(df_prices.index[:3])
df_prices.head()

Prices obtained: 21 days
Index([2026-02-12, 2026-02-13, 2026-02-17], dtype='object')


,Close,Volume
2026-02-12,309.152191,28194000
2026-02-13,305.804565,20236000
2026-02-17,302.606812,23750800
2026-02-18,303.726044,15847700
2026-02-19,303.346283,13448600


In [7]:
# 2. GOOG news via NewsAPI
from dotenv import load_dotenv
import os

load_dotenv()
newsapi = NewsApiClient(api_key=os.getenv("NEWSAPI_KEY"))

all_articles = []

# NewsAPI only allows 30 days back in the free tier
news = newsapi.get_everything(
    q="Google GOOG stock",
    language="en",
    sort_by="publishedAt",
    page_size=100
)

for article in news["articles"]:
    all_articles.append({
        "date": article["publishedAt"][:10],
        "title": article["title"],
        "description": article["description"]
    })

df_news = pd.DataFrame(all_articles)
df_news["date"] = pd.to_datetime(df_news["date"]).dt.date

print(f"News obtained: {len(df_news)}")
df_news.head()

News obtained: 18


,date,title,description
0,2026-03-09,"Dear Oracle Stock Fans, Mark Your Calendars fo...",Oracle stock is heading into a key earnings ev...
1,2026-03-08,Show HN: I logged Gemini's stock predictions f...,Article URL: https://huggingface.co/datasets/l...
2,2026-03-07,Alphabet: Apple AI Deal Is The Biggest Blind S...,Alphabet: Apple AI Deal Is The Biggest Blind S...
3,2026-03-05,Alphabet Stock Edges Lower in Early Trading as...,Alphabet Inc. shares dipped modestly in early ...
4,2026-03-05,Alphabet (GOOGL) is Continuing Its Development...,"Patient Capital Management, a value investing ..."


In [8]:
df_prices.to_csv("data/GOOG_prices.csv")
df_news.to_csv("data/GOOG_news.csv", index=False)

print(f"Prices: {len(df_prices)} days")
print(f"News: {len(df_news)} articles")

Prices: 21 days
News: 18 articles
